In [1]:
import pandas as pd
from collections import defaultdict


In [2]:
# 1) 데이터 로드
# text  : 메일 본문(문자열)
# label : 정답 라벨 ("spam" 또는 "ham")
df = pd.read_csv("../data/spam_mail.csv")

texts = df["text"].tolist()
labels = df["label"].tolist()

In [5]:
# 2) 토큰화 (공백 기준)
def tokenize(text: str) -> list[str]:
    # TODO: 공백 기준으로 split하고, 양쪽 공백 제거 + 빈 토큰 제거
    return [t.strip() for t in text.split() if t.strip()]


In [6]:
# 3) 단어 빈도 세기
# word_count[label][word] 형태로 라벨별 단어 등장 횟수를 저장
word_count = {
    "spam": defaultdict(int),
    "ham": defaultdict(int),
}
class_count = defaultdict(int)
class_total_words = defaultdict(int)

for text, label in zip(texts, labels):
    class_count[label] += 1
    tokens = tokenize(text)
    for w in tokens:
        word_count[label][w] += 1
        class_total_words[label] += 1

In [7]:
# 4) 사전확률(Prior)
# 사전확률은 아무 정보가 없을 때 스팸/정상일 기본 확률
# 예: 전체 메일이 100개이고 그중 스팸이 40개면 P(spam)=0.4
total_docs = len(labels)
prior = {
    "spam": class_count['spam'] / total_docs,  # TODO: P(spam)
    "ham":  class_count['ham'] / total_docs,  # TODO: P(ham)
}

In [8]:
# 5) 조건부확률 P(word | class)
# 나이브 베이즈 분류에서는 문장에 등장한 단어들에 대해 P(word|spam), P(word|ham)을 곱해서 스팸일 가능성을 계산합니다.
# 여기서는 아주 단순히
# P(word|label) = (label에서 word가 나온 횟수) / (label의 전체 단어 수)
# 단, 학습 데이터에 한 번도 없던 단어는 확률이 0이 되어 전체 곱이 0이 되는 문제가 생기므로 
# 아주 작은 확률(1e-6)로 처리합니다.
def word_prob(word: str, label: str) -> float:
    # TODO: word가 해당 label에서 한 번도 안 나왔으면 1e-6 반환
    if word_count[label][word] == 0:
        return 1e-6
    return word_count[label][word] / class_total_words[label]

In [9]:
# 6) 중간 과정 출력 예측 함수
# spam_score = P(spam) * Π P(word|spam)
# ham_score  = P(ham)  * Π P(word|ham)
# 위 두 값을 비교해서 더 큰 쪽을 예측 결과로 선택
def predict_verbose(text: str) -> str:
    tokens = tokenize(text)

    spam_prob = prior["spam"]
    ham_prob = prior["ham"]

    print(f"문장: {text}")
    print(f"초기 확률 -> P(spam)={spam_prob:.6f}, P(ham)={ham_prob:.6f}")
    print("-" * 60)

    for w in tokens:
        p_w_spam = word_prob(w, "spam")
        p_w_ham = word_prob(w, "ham")

        print(f"단어: '{w}'")
        print(f"  P({w}|spam) = {p_w_spam:.6f}")
        print(f"  P({w}|ham)  = {p_w_ham:.6f}")

        # TODO: 누적 확률 업데이트
        spam_prob *= p_w_spam
        ham_prob  *= p_w_ham

        print(f"  누적 -> spam_prob = {spam_prob:.12e}")
        print(f"         ham_prob  = {ham_prob:.12e}")
        print("-" * 60)

    print(f"최종 비교 -> spam={spam_prob:.12e} vs ham={ham_prob:.12e}")

    # TODO: 더 큰 확률의 클래스를 반환
    if spam_prob > ham_prob:
        print("=> 예측 결과: spam")
        return "spam"
    else:
        print("=> 예측 결과: ham")
        return "ham"

In [10]:
# 7) 테스트
test_samples = [
    ("지금 무료 할인 쿠폰 드립니다", "spam"),
    ("내일 회의 일정 공유드립니다", "ham"),
    ("대출 즉시 승인 가능합니다", "spam"),
    ("보고서 수정해서 보내주세요", "ham"),
    ("당첨 이벤트 참여하세요", "spam"),
]

correct = 0
for text, true_label in test_samples:
    pred = predict_verbose(text)
    print(f"정답: {true_label}")
    print("=" * 60)
    if pred == true_label:
        correct += 1

print(f"정확도: {correct / len(test_samples):.2f}")

문장: 지금 무료 할인 쿠폰 드립니다
초기 확률 -> P(spam)=0.450000, P(ham)=0.550000
------------------------------------------------------------
단어: '지금'
  P(지금|spam) = 0.113636
  P(지금|ham)  = 0.000001
  누적 -> spam_prob = 5.113636363636e-02
         ham_prob  = 5.500000000000e-07
------------------------------------------------------------
단어: '무료'
  P(무료|spam) = 0.045455
  P(무료|ham)  = 0.000001
  누적 -> spam_prob = 2.324380165289e-03
         ham_prob  = 5.500000000000e-13
------------------------------------------------------------
단어: '할인'
  P(할인|spam) = 0.022727
  P(할인|ham)  = 0.000001
  누적 -> spam_prob = 5.282682193839e-05
         ham_prob  = 5.500000000000e-19
------------------------------------------------------------
단어: '쿠폰'
  P(쿠폰|spam) = 0.022727
  P(쿠폰|ham)  = 0.000001
  누적 -> spam_prob = 1.200609589509e-06
         ham_prob  = 5.500000000000e-25
------------------------------------------------------------
단어: '드립니다'
  P(드립니다|spam) = 0.045455
  P(드립니다|ham)  = 0.000001
  누적 -> spam_prob = 5.45